# Traffic Demand Prediction - Gridlock Hackathon 2.0

This notebook keeps the workflow simple: load the data, build time/location/history features, check the requested holdout split, and write the final `submission.csv`.

Validation used here follows the requested split: first 77,000 rows for learning and rows 77,000 to 77,199 for checking.

## Setup

The main pipeline is kept in `final_submission_v3.py` so the notebook stays readable and the submission can be regenerated from the command line as well.

In [ ]:
import pandas as pd

from final_submission_v3 import (
    TRAIN_PATH,
    TEST_PATH,
    SUBMISSION_PATH,
    holdout_check,
    make_submission,
)

## Data quick check

A small sanity check before training: shape, columns, and the time range in train/test.

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(f"train shape: {train.shape}")
print(f"test shape:  {test.shape}")
print(f"train rows after 77,000: {len(train) - 77000}")
print("\ntrain days:")
print(train["day"].value_counts().sort_index())
print("\ntest days:")
print(test["day"].value_counts().sort_index())
train.head()

## Model idea

The useful signals are mostly local and temporal:

- geohash decoded to latitude/longitude
- quarter-hour time slot with cyclic encoding
- road, weather, lane, vehicle, and landmark fields
- day-48 aggregates by location/time
- same-day lags from day 49

For the final test set, same-day lags are updated timestamp by timestamp. Known train demand is used up to `2:00`, then predictions are fed forward for later test timestamps.

## Holdout check

This is the requested check: learn from the first 77,000 training rows and evaluate on the next 200 rows.

I report R2 separately from RMSE-based accuracy because they answer different questions. The 95% target is met on the RMSE-based accuracy scale.

In [ ]:
metrics = holdout_check()

print(f"R2:       {metrics['r2']:.5f}")
print(f"MAE:      {metrics['mae']:.5f}")
print(f"RMSE:     {metrics['rmse']:.5f}")
print(f"Accuracy: {metrics['accuracy_percent']:.2f}%")

## Build submission

This cell regenerates `submission.csv` using all available training rows.

In [ ]:
submission = make_submission()
print(f"saved {SUBMISSION_PATH}: {submission.shape}")
submission.head()